In [1]:
import numpy as np
import opendssdirect as dss
import yadi.dss.model as dss_model
import yadi.dss.sensitivity as dss_sens
import yadi.dss.qsts as dss_qsts

In [2]:
#Load the case3_unbalanced file from PowerModelsDistribution.jl
case_name = 'case3_unbalanced'
cktfile = r"../test_cases/{case_name}.dss".format(case_name=case_name)

# Using yadi to get timeseries data of voltage magnitudes in per unit.

In [3]:
qsts = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step="15m",
    simulation_steps=24*4
    )
qsts.run()

#Nodal current injections timeseries
I = qsts.currents_mvts

#Complex power injection timeseries
S = qsts.complex_powers_mvts

#Voltage phasor timeseries
V = qsts.voltages_mvts

#NEW: VOLTAGE MAGNITUDE PER UNIT (PU) TIMESERIES
Vmags_pu = qsts.vmags_pu_mvts

/home/sam/Research/yadi/yadi/dss/qsts.py:230: UserWarning: QSTS has not been initialized. Initiailizing before run.
  warnings.warn("QSTS has not been initialized. Initiailizing before run.")


DSS Running file: ../test_cases/case3_unbalanced.dss
DSS Compiled Circuit: 3bus_example
DSS Running file: ../test_cases/case3_unbalanced.dss
DSS Compiled Circuit: 3bus_example
QSTS Initialized, Returned:  ['', '', '']


QSTS running...:   0%|          | 0/96 [00:00<?, ?it/s]

QSTS running...: 100%|██████████| 96/96 [00:00<00:00, 778.98it/s]


In [4]:
V

array([[ 229.99324947-5.63065938e-06j, -114.99662818-1.99179996e+02j,
        -114.99662201+1.99179999e+02j,  226.53589157-8.86663439e-01j,
        -114.63226923-1.97645725e+02j, -114.39629414+1.97162948e+02j,
         222.51339221-1.88062848e+00j, -114.19367479-1.95869768e+02j,
        -113.72107263+1.94814466e+02j],
       [ 229.99324947-5.63065938e-06j, -114.99662818-1.99179996e+02j,
        -114.99662201+1.99179999e+02j,  226.53589157-8.86663439e-01j,
        -114.63226923-1.97645725e+02j, -114.39629414+1.97162948e+02j,
         222.51339221-1.88062848e+00j, -114.19367479-1.95869768e+02j,
        -113.72107263+1.94814466e+02j],
       [ 229.99324947-5.63065938e-06j, -114.99662818-1.99179996e+02j,
        -114.99662201+1.99179999e+02j,  226.53589157-8.86663439e-01j,
        -114.63226923-1.97645725e+02j, -114.39629414+1.97162948e+02j,
         222.51339221-1.88062848e+00j, -114.19367479-1.95869768e+02j,
        -113.72107263+1.94814466e+02j],
       [ 229.99324947-5.63065938e-06j, -

## Display the voltage magnitude per unit time series

In [5]:
Vmags_pu

array([[0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],
       [0.99589998, 0.99589999, 0.99589999, 0.9809367 , 0.98935955,
        0.98703856, 0.96354566, 0.98175698, 0.97677875],


## Extract the base voltages for every node --- note that these are in kV.

In [6]:
kV_base = qsts.get_node_base_voltages()
kV_base

SOURCEBUS.1    0.23094
SOURCEBUS.2    0.23094
SOURCEBUS.3    0.23094
PRIMARY.1      0.23094
PRIMARY.2      0.23094
PRIMARY.3      0.23094
LOADBUS.1      0.23094
LOADBUS.2      0.23094
LOADBUS.3      0.23094
dtype: float64

In [7]:
V_base = kV_base*1000
V_base

SOURCEBUS.1    230.940108
SOURCEBUS.2    230.940108
SOURCEBUS.3    230.940108
PRIMARY.1      230.940108
PRIMARY.2      230.940108
PRIMARY.3      230.940108
LOADBUS.1      230.940108
LOADBUS.2      230.940108
LOADBUS.3      230.940108
dtype: float64

## Extract the node order in the Ybus.

In [8]:
dss.Circuit.YNodeOrder()

['SOURCEBUS.1',
 'SOURCEBUS.2',
 'SOURCEBUS.3',
 'PRIMARY.1',
 'PRIMARY.2',
 'PRIMARY.3',
 'LOADBUS.1',
 'LOADBUS.2',
 'LOADBUS.3']

## Lines 

In [9]:
dss.Lines.AllNames()

['ohline', 'quad']

In [114]:
import warnings

def make_phase_label(
        phase_number : int, # 1,2,3
        mapping = {
            1: 'a',
            2: 'b',
            3: 'c'
        }
):
    """
    Makes a phase label ('a','b','c') from a phase number (1,2,3)
    """
    if phase_number not in mapping.keys():
        if phase_number == 0:
            warnings.warn("Phase number is 0. Assuming you meant '1' -> 'a'")
            return mapping[1]
        else:    
            raise Exception("Invalid phase number. Must be in {1,2,3}")
    else:
        return mapping[phase_number]
    
def get_line_currents(structure="dict"):
    """
    Gets the lines currents in the system at the current timestep/solution. 
    The first key is always the line name.

    Parameters
    ----------
    structure : str, optional
        The structure of the output. The default is "dict".
        - If structure == "dict": 
            - Then the value is a dictionary, where:
            - The first key is the terminal number (1,2), 
                - i.e., whether to extract the current flowing from terminal 1->2 or 2->1.
            - The second key is the phase (a,b,c)
            - The value is the from-to current for that terminal and phase
        - If structure == "matrix":
            - Then the value is a 2x3 matrix, where:
            - The rows are the terminals (from or to) and the columns are the phases \in {a,b,c}
    Note that I_{n1,n2}^{(\phi)} != I_{n2,n1}^{(\phi)} in general.
    """
    
    network_currents = {}
    names_lines = dss.Lines.AllNames()
    line_idx,line = 0,dss.Lines.First()

    while line: #iterate over lines
        line_label = names_lines[line_idx] #get name of line
        R2n_line_currents = np.asarray(dss.CktElement.Currents()) #get network_currents at this line (R^{2n})
        line_currents = R2n_line_currents[0::2] + 1j*R2n_line_currents[1::2] #convert to complex
        num_terminals = dss.CktElement.NumTerminals() #get number of terminals
        num_phases = dss.Lines.Phases() #get number of phases

        # Check that the number of terminals and phases are valid
        if num_terminals < 2:
            raise Exception("There is a floating line, check that each line has two terminals.")
        if num_phases == 0:
            raise Exception("Invalid line specification. There are no phases in this line.")

        # Get line currents in the appropriate structure
        if structure == "matrix":
            f_currents = line_currents[0:num_phases]
            t_currents = line_currents[num_phases:]        
            # 2xnum_phases matrix, where the rows are the terminals (from or to) and the columns are the phases \in {a,b,c}
            network_currents[line_label] = np.vstack((f_currents,t_currents)) 
        elif structure == "dict":
            terminal_currents = {} # dictionary of dictionaries of phase currents for each terminal
            for term_idx in range(num_terminals): #terminal index (0,1)
                phase_currents = {} # dictionary of currents by phase for the current terminal terminal
                terminal_label = str(term_idx+1) #terminal label (1,2)
                for ph_idx in range(num_phases): #phase index (0,1,2)
                    phase_number = dss.CktElement.NodeOrder()[ph_idx] #phase number: (0,1,2) -> (1,2,3)
                    phase_label =  make_phase_label(phase_number) #phase_label: (1,2,3) -> (a,b,c)
                    phase_currents[phase_label] = line_currents[term_idx*num_phases+ph_idx]
                terminal_currents[terminal_label] = phase_currents
            network_currents[line_label] = terminal_currents
        else:
            raise Exception("Invalid structure. Must be 'matrix', 'dict', or 'dict2'")

        line = dss.Lines.Next() #increment line
        line_idx += 1 #increment index

    return network_currents


def get_line_ratings():
    """
    Returns a dictionary of the nominal and emergency ratings for each line.
    """
    ratings_lines = {}
    names_lines = dss.Lines.AllNames()
    line_idx,line = 0,dss.Lines.First()
    while line:
        name_line = names_lines[line_idx]
        # Get line ratings
        line_ratings = {
            'NormAmps': dss.Lines.NormAmps(), #normal ampere rating
            'EmergAmps': dss.Lines.EmergAmps(), #emergency ampere rating
        }
        ratings_lines[name_line] = line_ratings # Save the ratings for this line
        line = dss.Lines.Next() #increment line
        line_idx += 1 #increment index
    return ratings_lines


def get_line_data():
    """
    Returns dictionaries of line data, specifically:
        -BusNames: Array of strings. Get Bus definitions to which each terminal is connected. 0-based array.
        -NumTerminals: Number of Terminals this Circuit Element
        -NumConductors: Number of Conductors per Terminal
        -NodeOrder: Array of integer containing the node numbers (representing phases, for example) for each conductor of each terminal.
    """
    data_lines = {}
    names_lines = dss.Lines.AllNames()
    line_idx,line = 0,dss.Lines.First()
    while line:
        name_line = names_lines[line_idx] #get name of line
        # Get line data
        line_data = {
            'BusNames': dss.CktElement.BusNames(),
            'NumTerminals': dss.CktElement.NumTerminals(),
            'NumConductors': dss.CktElement.NumConductors(),
            'NodeOrder': dss.CktElement.NodeOrder(),
            'Phases': dss.Lines.Phases(), #number of phases
            'NormAmps': dss.Lines.NormAmps(), #normal ampere rating
            'EmergAmps': dss.Lines.EmergAmps(), #emergency ampere rating
        }
        data_lines[name_line] = line_data # Save the data for this line      
        line = dss.Lines.Next() #increment line
        line_idx += 1 #increment index
    return data_lines


In [121]:
get_line_ratings()

{'ohline': {'NormAmps': 400.0, 'EmergAmps': 600.0},
 'quad': {'NormAmps': 400.0, 'EmergAmps': 600.0}}

In [122]:
get_line_currents('dict')

{'ohline': {'1': {'a': (40.330202694084846-13.815950651558865j),
   'b': (-24.753279561465206-16.201215924521193j),
   'c': (-1.9299133639598267+29.671987131597234j)},
  '2': {'a': (-40.330195600732395+13.819602884686788j),
   'b': (24.756454167232732+16.199378893341645j),
   'c': (1.9267426203842888-29.673822274926806j)}},
 'quad': {'1': {'a': (40.33019560073262-13.819602884687015j),
   'b': (-24.75645416723347-16.199378893341645j),
   'c': (-1.9267426203851983+29.673822274926806j)},
  '2': {'a': (-40.3301734623974+13.823195278957542j),
   'b': (24.759602291182148+16.19754828578948j),
   'c': (1.9236068010732197-29.67564721386134j)}}}

In [123]:
get_line_currents('matrix')

{'ohline': array([[ 40.33020269-13.81595065j, -24.75327956-16.20121592j,
          -1.92991336+29.67198713j],
        [-40.3301956 +13.81960288j,  24.75645417+16.19937889j,
           1.92674262-29.67382227j]]),
 'quad': array([[ 40.3301956 -13.81960288j, -24.75645417-16.19937889j,
          -1.92674262+29.67382227j],
        [-40.33017346+13.82319528j,  24.75960229+16.19754829j,
           1.9236068 -29.67564721j]])}

In [125]:
dss.Transformers.First()
dss.Transformers.AllNames()


[]

In [77]:
currents = get_line_currents()

Line.ohline
Line.quad


ValueError: inconsistent shapes

In [52]:
#currents[0::2] +  1j*currents[1::2]


In [78]:
dss.Lines.All

AttributeError: module 'opendssdirect.Lines' has no attribute 'Currents'

In [37]:
dss.Lines.AllNames()

['ohline', 'quad']

In [91]:
dss.Lines.First()
all_curr_vals = dss.PDElements.AllCurrents()
#n_currents_per_line = 
len(all_curr_vals)/2/2

6.0

In [38]:
dss.Circuit.AllBusNames()

['sourcebus', 'primary', 'loadbus']